In [1]:
# !pip install pandas

In [2]:
import pandas as pd

In [3]:
import glob
import os

# Find all files in the raw questions directory
raw_files = glob.glob('../../ttct/data/raw/*.csv')

question_to_group = {}

for file_path in raw_files:
    group_name = os.path.basename(file_path)
    tmp_df = pd.read_csv(file_path)
    for q in tmp_df['CoT_prompt'].tolist():
        question_to_group[q] = group_name
# question_to_group is now a dictionary mapping each question to its group (filename)

In [4]:
# tmp_df

In [5]:
len(question_to_group)

697

In [6]:
# group_keys

In [7]:
import pandas as pd
import glob
import random

# Load all evaluation files for gpt-4.1
files = glob.glob('../../ttct/data/evaluations/temp_1/gpt-4.1-2025-04-14.csv')
df_list = [pd.read_csv(f) for f in files]
df = pd.concat(df_list, ignore_index=True)

# Keep only the required columns
df = df[['infer_cot_input', 'infer_cot_pred', 'eval_cot_input', 'eval_cot_pred']]

# Group rows by the prefix of 'infer_cot_input' (first sentence or instruction)
def get_group_key(text):
    # return text.split('.')[0].strip()  # adjust if instructions are not sentence-based
    # return text.strip()[:20]
    if text in question_to_group:
        return question_to_group[text]
    else:     
        return "unknown"

df['group'] = df['infer_cot_input'].apply(get_group_key)
group_keys = df['group'].unique()

# Sample 5 rows per group
sampled_rows = []
for key in group_keys:
    if key == "unknown":
        continue
    group_df = df[df['group'] == key]
    sampled = group_df.sample(n=min(5, len(group_df)), random_state=42)
    sampled_rows.append(sampled)

sampled_df = pd.concat(sampled_rows).drop(columns=['group']).reset_index(drop=True)

# Save or display the sampled dataframe
# print(sampled_df)
sampled_df.shape

(35, 4)

In [8]:
# Load the dataframe from the specified CSV file
df_gpt4 = pd.read_csv('../../ttct/data/evaluations/temp_1/gpt-4.1-2025-04-14.csv')

# Generate mapping from row index to group name using get_group_key
row_to_group = {idx: get_group_key(row['infer_cot_input']) for idx, row in df_gpt4.iterrows()}
len(row_to_group)

700

In [9]:
# row_to_group

In [11]:
# eval_dir = '../../ttct/data/evaluations/temp_1'
# csv_files = glob.glob(os.path.join(eval_dir, '*.csv'))

# for file in csv_files:
#     df_tmp = pd.read_csv(file)
#     print(f"{os.path.basename(file)}: {df_tmp.shape}")

In [12]:
def sample_evaluation_file_sequential(filepath, n_per_group=5, group_size=100, random_state=42):
    print(f"Sampling from {filepath}")
    df = pd.read_csv(filepath)
    df['group'] = [row_to_group[i] for i in df.index]
    sampled_rows = []
    for group_name in df['group'].unique():
        group_df = df[df['group'] == group_name]
        sampled = group_df.sample(n=min(n_per_group, len(group_df)), random_state=random_state)
        sampled_rows.append(sampled)
    sampled_df = pd.concat(sampled_rows).reset_index(drop=True)
    return sampled_df


In [13]:
sample_evaluation_file_sequential('../../ttct/data/evaluations/temp_1/Qwen2.5-72B-Instruct.csv',).shape

Sampling from ../../ttct/data/evaluations/temp_1/Qwen2.5-72B-Instruct.csv


(35, 11)

In [14]:
# List of evaluation files to sample from
eval_files = [
    '../../ttct/data/evaluations/temp_1/Qwen2.5-72B-Instruct.csv',
    '../../ttct/data/evaluations/temp_1/gpt-4.1-2025-04-14.csv',
    '../../ttct/data/evaluations/temp_1/OLMo-2-1124-13B-Instruct.csv'
]

# Sample 35 rows per file and concatenate results
sampled_all = []
for filepath in eval_files:
    sampled = sample_evaluation_file_sequential(filepath, n_per_group=5)
    sampled['source_file'] = os.path.basename(filepath)
    print(f"Sampled {sampled.shape[0]} rows from {os.path.basename(filepath)}")
    sampled_all.append(sampled)

big_table = pd.concat(sampled_all, ignore_index=True)
big_table.shape  # Should be (105, columns)

Sampling from ../../ttct/data/evaluations/temp_1/Qwen2.5-72B-Instruct.csv
Sampled 35 rows from Qwen2.5-72B-Instruct.csv
Sampling from ../../ttct/data/evaluations/temp_1/gpt-4.1-2025-04-14.csv
Sampled 35 rows from gpt-4.1-2025-04-14.csv
Sampling from ../../ttct/data/evaluations/temp_1/OLMo-2-1124-13B-Instruct.csv
Sampled 35 rows from OLMo-2-1124-13B-Instruct.csv


(105, 15)

In [15]:
group_keys

array(['common_problem_task_prompt.csv', 'consequences_task_prompt.csv',
       'im_task_prompt.csv', 'is_task_prompt.csv', 'js_task_prompt.csv',
       'situation_task_prompt.csv', 'unusual_task_prompt.csv'],
      dtype=object)

In [17]:
sampled_df.head(10)

,infer_cot_input,infer_cot_pred,eval_cot_input,eval_cot_pred
0,"Common Problems Task. In this task, you will b...",Absolutely! Let’s break down the scenario of p...,You are an expert of psychology. Your objectiv...,Fluency: 5 Flexibility: 5 Originality: 4 Elab...
1,"Common Problems Task. In this task, you will b...",Absolutely! Let’s break down the scenario: **W...,You are an expert of psychology. Your objectiv...,Fluency: 5 \n\nFluency: 5 \nFlexibility: 5 \n...
2,"Common Problems Task. In this task, you will b...",Certainly! Let’s break down the scenario: **Ov...,You are an expert of psychology. Your objectiv...,Fluency: 5\nFlexibility: 4\nOriginality: 4\nE...
3,"Common Problems Task. In this task, you will b...",Certainly! Organizing an international peace s...,You are an expert of psychology. Your objectiv...,"Fluency: 5, Flexibility: 5, Originality: 4, E..."
4,"Common Problems Task. In this task, you will b...",Sure! Let's break down the scenario of **start...,You are an expert of psychology. Your objectiv...,"Fluency, Flexibility, Originality, Elaboratio..."
5,What would happen if the world's deserts sudde...,Absolutely—let’s break down the possible conse...,You are an expert of psychology. Your objectiv...,Fluency: 5\nFlexibility: 5\nOriginality: 5\nE...
6,What would happen if plastic could biodegrade ...,"Absolutely, let's break this down step by step...",You are an expert of psychology. Your objectiv...,Fluency: 4 Flexibility: 4 Originality: 4 E...
7,What would happen if the speed of light was re...,"Great question! Changing the speed of light, *...",You are an expert of psychology. Your objectiv...,Fluency: 4 \nFlexibility: 4 \nOriginality: 4 ...
8,What might happen if people could change their...,Absolutely! Let’s break down the possible impl...,You are an expert of psychology. Your objectiv...,1. Fluency (4) 2. Flexibility (5) 3. Originali...
9,What are the consequences if everyone on Earth...,"Absolutely, let's break it down step by step:\...",You are an expert of psychology. Your objectiv...,Flexibility: 4 \nOriginality: 5 \nElaboration...


In [19]:
print(sampled_df['infer_cot_pred'].values[0])

Absolutely! Let’s break down the scenario of pioneering a mission to the bottom of the ocean and identify as many potential problems or issues as possible, step by step:

**1. Technical Challenges**  
- **Extreme water pressure:** The pressure at the bottom of the ocean (10,000+ meters) is crushing, requiring specially engineered vehicles and materials.
- **Engineering submersibles:** Building submersibles tough enough to survive the pressure while remaining maneuverable and lightweight.
- **Communication limitations:** Radio waves don't travel well through water, so maintaining contact with the surface or mission control is difficult.
- **Power supply:** Supplying and storing sufficient power for propulsion, life support, lights, and data-gathering equipment over potentially extended missions.
- **Navigation:** GPS does not work underwater; alternative navigation methods (acoustic, inertial) are required but less accurate.
- **Equipment failure:** The harsh conditions may cause mechan